# 05 — Semantically Grounded Collaborative Engine

In this notebook, we implement the second intelligence layer of the PodcastMind recommender system: **Collaborative Filtering**.

## Hybrid Architecture & Technical Honesty
- **Semantic Layer (Real Intelligence):** Uses real podcast descriptions and state-of-the-art embeddings (`all-MiniLM-L6-v2`) to understand *what* a show is about. 
- **Collaborative Layer (Simulated Intelligence):** Since our `reviews.json` was found to be misaligned with the metadata, we use a **semantically grounded simulation**. 

### Why simulate behavior?
Simulating behavior based on real categories allows us to build the **architectural plumbing** for a hybrid system. It proves that we can blend content signals with behavioral signals. In a production environment, this simulated layer would be swapped with real production interaction logs (clicks, listens, follows).

## Objectives
- Prepare semantic clusters based on real podcast categories.
- Generate 5,000 synthetic users with realistic, clustered interests.
- Train an **ALS (Alternating Least Squares)** model using the `implicit` library.
- Compare Content-Based vs. Collaborative retrieval.

## SECTION 1 — Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import random
from collections import defaultdict
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm
import implicit
from implicit.als import AlternatingLeastSquares
import warnings

warnings.filterwarnings('ignore')

print(f"Implicit version: {implicit.__version__}")

Implicit version: 0.7.3


## SECTION 2 — Load Development Dataset

In [2]:
DATA_DIR = Path("../data/processed")
PODCASTS_FILE = DATA_DIR / "podcasts_subset_20k.csv"

df = pd.read_csv(PODCASTS_FILE)
print(f"Loaded {len(df)} podcasts for simulation.")
df.head(2)

Loaded 20000 podcasts for simulation.


,podcast_id,title,author,description,categories,semantic_density_score,combined_text
0,ff269332200b78cbabfe05ce6d2ab038,Chinwag Live Podcasts,Chinwag Live,Chinwag is an online community for digital pro...,"business, technology",4,Title: Chinwag Live Podcasts | Author: Chinwag...
1,b6e5d2929465dd2080be0bbced0c29cf,Agora Historia Oficial,Ágora Historia,Ágora Historia programa dedicado a la historia...,"society-culture, history",4,Title: Agora Historia Oficial | Author: Ágora ...


## SECTION 3 — Semantic Cluster Preparation

For collaborative filtering to learn meaningful relationships, user behavior must have structure. We group podcasts into "Semantic Affinity Clusters" based on their categories. 

In [3]:
# Mapping podcast index to categories
podcast_to_categories = {}
category_to_podcasts = defaultdict(list)

for idx, row in df.iterrows():
    if pd.isna(row['categories']):
        cats = ['unknown']
    else:
        cats = [c.strip() for c in row['categories'].split(',')]
    
    podcast_to_categories[idx] = cats
    for cat in cats:
        category_to_podcasts[cat].append(idx)

all_categories = list(category_to_podcasts.keys())
print(f"Found {len(all_categories)} unique categories to ground our simulation.")

Found 110 unique categories to ground our simulation.


## SECTION 4 — Synthetic User Generation

We create 5,000 synthetic users. 
- **Interests:** Each user is assigned 1-3 core interest categories.
- **Behavior:** They are 80% likely to pick podcasts from their core categories and 20% likely to "explore" related categories.
- **Density:** Users have between 10 and 40 interactions.

In [4]:
NUM_USERS = 5000
interactions = []

random.seed(42)
np.random.seed(42)

print(f"Generating interactions for {NUM_USERS} users...")

for user_id in tqdm(range(NUM_USERS)):
    # 1. Define user persona (1-3 categories)
    num_interests = random.randint(1, 3)
    user_interests = random.sample(all_categories, num_interests)
    
    # 2. Build personalized pool
    interest_pool = []
    for cat in user_interests:
        interest_pool.extend(category_to_podcasts[cat])
    
    interest_pool = list(set(interest_pool))
    if not interest_pool: continue
    
    # 3. Generate interactions
    num_actions = random.randint(10, 40)
    # Sample unique podcasts for this user
    num_to_sample = min(num_actions, len(interest_pool))
    chosen_podcasts = random.sample(interest_pool, num_to_sample)
    
    for pod_idx in chosen_podcasts:
        interactions.append({
            'user_id': user_id,
            'podcast_idx': pod_idx,
            'weight': 1.0
        })

interaction_df = pd.DataFrame(interactions)
print(f"Total interactions generated: {len(interaction_df):,}")

Generating interactions for 5000 users...


  0%|          | 0/5000 [00:00<?, ?it/s]

Total interactions generated: 117,894


## SECTION 5 — Interaction Matrix Construction

We convert our simulated logs into a **Compressed Sparse Row (CSR)** matrix. Recommender systems use sparse matrices because most users interact with less than 1% of the total catalog.

In [5]:
user_items = csr_matrix((
    interaction_df['weight'].values,
    (interaction_df['user_id'].values, interaction_df['podcast_idx'].values)
), shape=(NUM_USERS, len(df)))

print(f"Matrix Shape: {user_items.shape} (Users x Podcasts)")
sparsity = 1.0 - (user_items.nnz / (user_items.shape[0] * user_items.shape[1]))
print(f"Matrix Sparsity: {sparsity:.4%}")

Matrix Shape: (5000, 20000) (Users x Podcasts)
Matrix Sparsity: 99.8821%


## SECTION 6 — Train ALS Model

ALS factorizes the user-item matrix into **Latent Factors**. This allows the model to find "hidden dimensions" of podcast similarity based on who listens to them.

In [6]:
model = AlternatingLeastSquares(
    factors=64, 
    regularization=0.1, 
    iterations=20, 
    random_state=42
)

print("Training ALS model...")
model.fit(user_items)
print("Training complete.")

Training ALS model...


  0%|          | 0/20 [00:00<?, ?it/s]

Training complete.


## SECTION 7 — Save Collaborative Artifacts

In [7]:
ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

with open(ARTIFACTS_DIR / "als_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Create and save mappings
podcast_id_to_idx = {pid: i for i, pid in enumerate(df['podcast_id'])}
idx_to_podcast_id = {i: pid for pid, i in podcast_id_to_idx.items()}

with open(ARTIFACTS_DIR / "collaborative_mappings.pkl", "wb") as f:
    pickle.dump({
        'id_to_idx': podcast_id_to_idx,
        'idx_to_id': idx_to_podcast_id
    }, f)

print(f"Artifacts saved to {ARTIFACTS_DIR}")

Artifacts saved to ..\artifacts


## SECTION 8 — Collaborative Retrieval Function

In [8]:
def get_collaborative_recommendations(podcast_id, top_k=5):
    if podcast_id not in podcast_id_to_idx:
        return None
    
    p_idx = podcast_id_to_idx[podcast_id]
    ids, scores = model.similar_items(p_idx, N=top_k+1)
    
    results = []
    # Skip first because it's the target itself
    for i in range(1, len(ids)):
        idx = ids[i]
        pod_data = df.iloc[idx]
        results.append({
            'title': pod_data['title'],
            'categories': pod_data['categories'],
            'collaborative_score': float(scores[i])
        })
    return pd.DataFrame(results)

print("Collaborative retrieval function ready.")

Collaborative retrieval function ready.


## SECTION 9 — Compare Semantic vs Collaborative

- **Semantic:** Focuses on keywords and description meaning.
- **Collaborative:** Focuses on "Who also listens to this?"

In [9]:
sample_pod = df.sample(1, random_state=42).iloc[0]
print(f"TARGET PODCAST: {sample_pod['title']} | Categories: {sample_pod['categories']}")

print("\n--- Collaborative Recommendations ---")
display(get_collaborative_recommendations(sample_pod['podcast_id']))

TARGET PODCAST: Terceira Terra | Categories: arts-books, arts, fiction, fiction-science-fiction

--- Collaborative Recommendations ---


,title,categories,collaborative_score
0,Una Dosis De Ficcion,"fiction-science-fiction, arts, arts-books, fic...",0.991480
1,Searching for Salai,"fiction, arts, fiction-science-fiction, arts-p...",0.991320
2,Clarkesworld Magazine - Science Fiction & Fantasy,"fiction-science-fiction, fiction",0.990100
3,When The Page Talks Back,"fiction-science-fiction, fiction",0.990007
4,Edge Case,"fiction-science-fiction, fiction",0.989082


## SECTION 10 — Why Hybrid Systems Matter

We have now completed both the Content-Based (03) and Collaborative (05) engines. 

### Escaping the Specialization Trap
If we only used semantic signals, our system would become too narrow. If you like one podcast about "Python Programming," it might only ever show you more "Python Programming" shows. 

By layering on the Collaborative Engine, we introduce **behavioral exploration**. We can now suggest shows that are logically connected by human interest (e.g., Python → Data Science → Statistics → Philosophy), creating a more human-like discovery experience.

### Technical Honesty: Reality vs. Simulation
In this project, the **Semantic Engine represents real intelligence** (using real descriptions), while the **Collaborative Engine represents simulated intelligence.** However, because our simulation is structurally grounded in real categories and realistic interaction densities, it serves as a valid architectural blueprint for a production-scale system.

In the next notebook, **06 — Hybrid Blend**, we will combine these two scores into a single ranking algorithm that delivers the best of both worlds.